In [1]:
import pandas as pd
import joblib

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

### Load the test data

In [3]:
X_train = pd.read_csv(r"../DATA/ModelData/X_train_selected.csv")
X_test = pd.read_csv(r"../DATA/ModelData/X_test_selected.csv")

y_train = pd.read_csv(r"../DATA/ModelData/y_train_selected.csv").squeeze()
y_test = pd.read_csv(r"../DATA/ModelData/y_test_selected.csv").squeeze()

### Load the four models

In [4]:
logistic_model = joblib.load("../models/logistic_regression.pkl")
decision_tree = joblib.load("../models/decision_tree.pkl")
random_forest = joblib.load("../models/random_forest.pkl")
gradient_boosting = joblib.load("../models/gradient_boosting.pkl")

### Load the scaler

In [5]:
scaler = joblib.load("../models/scaler.pkl")

### Scale test data for Logistic Regression

In [6]:
X_test_scaled = scaler.transform(X_test)

### Generate predictions

In [7]:
y_pred_lr = logistic_model.predict(X_test_scaled)
y_pred_dt = decision_tree.predict(X_test)
y_pred_rf = random_forest.predict(X_test)
y_pred_gb = gradient_boosting.predict(X_test)

### Compare model performance

In [8]:
models = {
    "Logistic Regression": y_pred_lr,
    "Decision Tree": y_pred_dt,
    "Random Forest": y_pred_rf,
    "Gradient Boosting": y_pred_gb
}

results = []

for name, pred in models.items():
    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, pred),
        "Precision": precision_score(y_test, pred, average="weighted"),
        "Recall": recall_score(y_test, pred, average="weighted"),
        "F1 Score": f1_score(y_test, pred, average="weighted")
    })

results_df = pd.DataFrame(results)

results_df

,Model,Accuracy,Precision,Recall,F1 Score
0,Logistic Regression,0.569250,0.550854,0.569250,0.515198
1,Decision Tree,0.473131,0.475204,0.473131,0.474122
2,Random Forest,0.574741,0.546822,0.574741,0.528677
3,Gradient Boosting,0.579913,0.557415,0.579913,0.529008


### Classification reports

In [9]:
for name, pred in models.items():
    print(f"\n{name}")
    print(classification_report(y_test, pred))


Logistic Regression
              precision    recall  f1-score   support

         <30       0.53      0.02      0.03      2122
         >30       0.49      0.33      0.40      6655
          NO       0.59      0.85      0.70      9981

    accuracy                           0.57     18758
   macro avg       0.54      0.40      0.37     18758
weighted avg       0.55      0.57      0.52     18758


Decision Tree
              precision    recall  f1-score   support

         <30       0.16      0.16      0.16      2122
         >30       0.40      0.41      0.40      6655
          NO       0.59      0.58      0.59      9981

    accuracy                           0.47     18758
   macro avg       0.38      0.38      0.38     18758
weighted avg       0.48      0.47      0.47     18758


Random Forest
              precision    recall  f1-score   support

         <30       0.43      0.02      0.03      2122
         >30       0.49      0.39      0.43      6655
          NO       0.61 

### Confusion matrices

In [10]:
for name, pred in models.items():
    print(f"\n{name}")
    print(confusion_matrix(y_test, pred))


Logistic Regression
[[  34  735 1353]
 [  15 2206 4434]
 [  15 1528 8438]]

Decision Tree
[[ 339  923  860]
 [ 805 2721 3129]
 [ 997 3169 5815]]

Random Forest
[[  34  858 1230]
 [  32 2589 4034]
 [  13 1810 8158]]

Gradient Boosting
[[  43  823 1256]
 [  30 2388 4237]
 [  14 1520 8447]]


### Identify the best model.

In [12]:
results_df.sort_values(
    by="F1 Score",
    ascending=False
)

,Model,Accuracy,Precision,Recall,F1 Score
3,Gradient Boosting,0.579913,0.557415,0.579913,0.529008
2,Random Forest,0.574741,0.546822,0.574741,0.528677
0,Logistic Regression,0.569250,0.550854,0.569250,0.515198
1,Decision Tree,0.473131,0.475204,0.473131,0.474122


### Save evaluation results

In [14]:
results_df.to_csv(
    "../DATA/model_evaluation_results.csv",
    index=False
)

### Save the best model

In [15]:
best_model_name = results_df.loc[
    results_df["F1 Score"].idxmax(), "Model"
]

print("Best Model:", best_model_name)

Best Model: Gradient Boosting


In [16]:
best_model = {
    "Logistic Regression": logistic_model,
    "Decision Tree": decision_tree,
    "Random Forest": random_forest,
    "Gradient Boosting": gradient_boosting
}[best_model_name]

In [17]:
joblib.dump(
    best_model,
    "../models/model.pkl"
)

['../models/model.pkl']